<a href="https://colab.research.google.com/github/noecastilloz/analisis_productos_sql/blob/main/An%C3%A1lisis_Productos_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación de las extensiones para ejecutar **SQL**

In [ ]:
# 1. Instalar jupysql
!pip install jupysql --quiet

# 2. Cargar la extensión SQL en el cuaderno
%load_ext sql

# 3. Conectar a la base de datos SQLite
%sql sqlite:///mi_base.db

Connecting to 'sqlite:///mi_base.db'

In [ ]:
import pandas as pd
import sqlite3

# 1. Leer el archivo CSV
df = pd.read_csv('/content/products-10000.csv')

# 2. Conectar a mi_base.db y guardar el DataFrame como la tabla 'productos'
conn = sqlite3.connect('mi_base.db')
df.to_sql('productos', conn, if_exists='replace', index=False)
conn.close()


In [ ]:
#Esto es solo una consulta de prueba para validar el funcionamiento del entorno
%%sql
SELECT *
FROM productos
LIMIT 5;

Running query in 'sqlite:///mi_base.db'

Index,Name,Description,Brand,Category,Price,Currency,Stock,EAN,Color,Size,Availability,Internal ID
1,Smart Fan Iron Cooker Go Wireless Portable,Catch enough role nearly.,Herman Ltd,Kids' Clothing,585,USD,194,3968600833473,Cornsilk,5x7 in,limited_stock,54
2,Fan,All movement yeah tax me.,"Braun, King and Rollins",Grooming Tools,992,USD,724,191126950284,Bisque,S,discontinued,49
3,Smart Speakerphone Charger Eco Plus Clean,Quickly inside pull line lay start.,Peck-Coleman,Fishing & Hunting,940,USD,769,7569143820621,Blue,Extra Large,pre_order,42
4,Premium Grill Trimmer Portable,Lawyer one than fire.,Hines Ltd,Skincare,324,USD,93,2705140928037,Ivory,50x70 cm,out_of_stock,93
5,Keyboard Freezer,Remain Congress blood plan voice.,"Spence, Webster and Orr",Laptops & Computers,908,USD,614,9830391008108,FloralWhite,10x10 cm,discontinued,91


# 🛠 **Estructura del proyecto**

Antes de iniciar el análisis se revisara la tabla para ver las columnas y conque tipo de datos contamos.

1. **Inspección de estructura y conteo de registros**

Exploración de la base de datos.

In [ ]:
# Muestra todas las columnas del DataFrame cargado
df.columns.tolist()

['Index',
 'Name',
 'Description',
 'Brand',
 'Category',
 'Price',
 'Currency',
 'Stock',
 'EAN',
 'Color',
 'Size',
 'Availability',
 'Internal ID']

In [ ]:
# Datos disponibles
%%sql
PRAGMA table_info(productos);

Running query in 'sqlite:///mi_base.db'

cid,name,type,notnull,dflt_value,pk
0,Index,INTEGER,0,None,0
1,Name,TEXT,0,None,0
2,Description,TEXT,0,None,0
3,Brand,TEXT,0,None,0
4,Category,TEXT,0,None,0
5,Price,INTEGER,0,None,0
6,Currency,TEXT,0,None,0
7,Stock,INTEGER,0,None,0
8,EAN,INTEGER,0,None,0
9,Color,TEXT,0,None,0


In [ ]:
# Total de registros
%%sql
SELECT COUNT(*) AS total_registros
FROM productos;

Running query in 'sqlite:///mi_base.db'

total_registros
10000


# 2. **Análisis exploratorio y preguntas de negocio**

1. Conteo de marcas y categorias únicas.

In [ ]:
%%sql
SELECT
    COUNT(DISTINCT Brand) AS total_marcas,
    COUNT(DISTINCT Category) AS total_categorias
FROM productos;

Running query in 'sqlite:///mi_base.db'

total_marcas,total_categorias
9241,34


In [ ]:
%%sql
SELECT
    Category,
    SUM(Stock) AS stock_total,
    ROUND(AVG(Price), 2) AS precio_promedio
FROM productos
GROUP BY Category
ORDER BY stock_total DESC
LIMIT 5;

Running query in 'sqlite:///mi_base.db'

Category,stock_total,precio_promedio
Clothing & Apparel,174236,484.99
Health & Wellness,160698,480.19
Camping & Hiking,160376,487.12
Team Sports,159781,501.57
Beauty & Personal Care,159161,518.56


2. Rango de precios (Min, Max y Promedio global).

In [ ]:
%%sql
SELECT
    MIN(Price) AS precio_minimo,
    MAX(Price) AS precio_maximo,
    ROUND(AVG(Price), 2) AS precio_promedio
FROM productos;

Running query in 'sqlite:///mi_base.db'

precio_minimo,precio_maximo,precio_promedio
1,999,503.37


3. Clasificación por rango de precio.

In [ ]:
%%sql
SELECT
    CASE
        WHEN Price < 100 THEN 'Económico (< $100)'
        WHEN Price BETWEEN 100 AND 500 THEN 'Gama Media ($100 - $500)'
        ELSE 'Gama Alta (> $500)'
    END AS segmento_precio,
    COUNT(*) AS total_productos,
    SUM(Stock) AS stock_total
FROM productos
GROUP BY segmento_precio
ORDER BY total_productos DESC;

Running query in 'sqlite:///mi_base.db'

segmento_precio,total_productos,stock_total
Gama Alta (> $500),5049,2526767
Gama Media ($100 - $500),3958,1984654
Económico (< $100),993,481159


4. Top 5 de productos más caros con su stock disponible.

In [ ]:
%%sql
SELECT
    Name,
    Category,
    Price,
    Stock
FROM productos
ORDER BY Price DESC
LIMIT 5;

Running query in 'sqlite:///mi_base.db'

Name,Category,Price,Stock
Portable Mixer Vacuum,Men's Clothing,999,842
Portable Speaker Powerbank Camera,Office Supplies,999,563
Charger Lamp,Men's Clothing,999,925
Mini Brush,Kitchen Appliances,999,256
Pro Clock Router Fridge Air Max,Shoes & Footwear,999,306


# 3. **Análisis Avanzado (Window Functions y Valoración de Inventario).**

1. Productos por encima del precio promedio de su categoría (Subconsulta / CTE).

In [ ]:
%%sql
WITH PromediosPorCategoria AS (
    SELECT Category, AVG(Price) AS avg_precio_categoria
    FROM productos
    GROUP BY Category
)
SELECT
    p.Name,
    p.Category,
    p.Price,
    ROUND(c.avg_precio_categoria, 2) AS precio_promedio_cat
FROM productos p
JOIN PromediosPorCategoria c ON p.Category = c.Category
WHERE p.Price > c.avg_precio_categoria
LIMIT 10;

Running query in 'sqlite:///mi_base.db'

Name,Category,Price,precio_promedio_cat
Smart Fan Iron Cooker Go Wireless Portable,Kids' Clothing,585,508.14
Fan,Grooming Tools,992,522.35
Smart Speakerphone Charger Eco Plus Clean,Fishing & Hunting,940,526.43
Keyboard Freezer,Laptops & Computers,908,537.45
Smart Fridge Plus,Kitchen Appliances,627,485.96
Compact Fan Scooter Headphones,Clothing & Apparel,870,484.99
Eco Dock Air,Beauty & Personal Care,602,518.56
Thermostat Stove Lamp,Kids' Clothing,707,508.14
Fan,Toys & Games,897,521.9
Wireless Light Heater Clock X,Grooming Tools,581,522.35


2. Ranking de los 3 productos más caros por categoría

In [ ]:
%%sql
WITH RankedProducts AS (
    SELECT
        Category,
        Name,
        Price,
        Stock,
        DENSE_RANK() OVER (PARTITION BY Category ORDER BY Price DESC) as ranking
    FROM productos
)
SELECT *
FROM RankedProducts
WHERE ranking <= 3
LIMIT 10;

Running query in 'sqlite:///mi_base.db'

Category,Name,Price,Stock,ranking
"Accessories (Bags, Hats, Belts)",Ultra Fan,992,143,1
"Accessories (Bags, Hats, Belts)",Ultra Dock,980,813,2
"Accessories (Bags, Hats, Belts)",Rechargeable Tablet Microphone Trimmer Plus,977,809,3
Automotive,Pro Vacuum Webcam Compact Advanced Wireless,994,923,1
Automotive,Eco Charger Stove Watch 360 Plus,993,95,2
Automotive,Smart Fan Phone Silent Sense Clean,989,291,3
Beauty & Personal Care,Smart Bicycle Microphone Touch,999,256,1
Beauty & Personal Care,Eco Dock Pro Air Go,998,680,2
Beauty & Personal Care,Smart Treadmill X,997,260,3
Bedding & Bath,Digital Cooler Watch Fan Eco Advanced,987,280,1


3. Valoración monetaria total del inventario por categoria.

In [ ]:
%%sql
SELECT
    Category,
    SUM(Stock) AS stock_total,
    ROUND(SUM(Price * Stock), 2) AS valor_inventario_total
FROM productos
GROUP BY Category
ORDER BY valor_inventario_total DESC
LIMIT 5;

Running query in 'sqlite:///mi_base.db'

Category,stock_total,valor_inventario_total
Clothing & Apparel,174236,87484993.0
Grooming Tools,158434,83238196.0
Headphones & Earbuds,156226,82866823.0
Beauty & Personal Care,159161,82032057.0
Team Sports,159781,79729718.0


4. Análisis de disponibilidad en stock.

In [ ]:
%%sql
SELECT
    Availability,
    COUNT(*) AS total_productos,
    SUM(Stock) AS stock_disponible
FROM productos
GROUP BY Availability
ORDER BY total_productos DESC;

Running query in 'sqlite:///mi_base.db'

Availability,total_productos,stock_disponible
discontinued,1706,854758
out_of_stock,1700,841244
pre_order,1673,847293
limited_stock,1644,837748
in_stock,1644,831464
backorder,1633,780073
